# 自动循迹搬运机器人

## 项目简介

本项目实现一个基于 Arduino、五路循迹模块、双电机底盘和双舵机夹爪的自动循迹搬运机器人。机器人先沿黑线从起点运行到取货站，识别宽标记后执行夹取动作，再继续循迹到卸货站并完成放置，形成一套完整的自动搬运流程。


## 主要器件

| 器件 | 数量 | 说明 |
| --- | --- | --- |
| Arduino Uno | 1 | 主控板 |
| 五路红外循迹模块 | 1 组 | 路径检测 |
| 双路电机驱动 | 1 | 控制左右轮 |
| 直流减速电机 | 2 | 底盘驱动 |
| 舵机 | 2 | 控制升降与夹爪 |
| 机械夹爪结构 | 1 套 | 抓取与放置物体 |


## 核心功能

- 五路循迹传感器进行路径跟随。
- 宽黑色标记识别为站点，用于取货和卸货定位。
- 双舵机执行夹爪张开、下降、夹紧、抬升和放置。
- 采用状态机区分去取货、抓取、去卸货、放置和任务完成。
- 串口输出传感器状态和站点计数，便于调试赛道。


## 引脚分配

| 模块 | 引脚 |
| --- | --- |
| 五路循迹传感器 | A0、A1、A2、A3、A4 |
| 左电机 PWM / 方向 | D5、D6、D7 |
| 右电机 PWM / 方向 | D9、D10、D11 |
| 升降舵机 | D3 |
| 夹爪舵机 | D4 |


## 接线说明

- 五路循迹模块的模拟或数字输出依次接 A0-A4，电源接 5V 和 GND。
- 左右电机通过驱动模块接到底盘，驱动的使能端接 PWM，方向端接数字口。
- 升降舵机接 D3，夹爪舵机接 D4，舵机建议独立 5V 供电并与主控共地。
- 赛道上需要预留足够宽的站点标记，让五路传感器能同时压线，便于可靠识别停靠点。


## 运行方式

1. 打开 `src/automatic_line_following_carrying_robot/automatic_line_following_carrying_robot.ino`。
2. 安装并连接循迹模块、双电机驱动和夹爪舵机。
3. 根据赛道颜色与传感器高度调整 `LINE_THRESHOLD`。
4. 根据站点位置调整 `PICKUP_STATION_INDEX` 与 `DROPOFF_STATION_INDEX`。
5. 上传程序后，先空载验证循迹，再放置物体测试抓取和搬运流程。


## 控制逻辑说明

- 主循环先读取五路循迹状态，再判断当前是否处于站点标记区。
- `handleMarker()` 只在标记上升沿累加站点编号，并加入冷却时间避免重复计数。
- 在 `FOLLOW_TO_PICKUP` 和 `FOLLOW_TO_DROPOFF` 两个状态下，小车按加权误差进行差速循迹。
- 到达取货站后执行 `performPickupSequence()`，完成张爪、下降、夹紧和抬升。
- 到达卸货站后执行 `performDropoffSequence()`，完成下降、放置和复位，随后任务结束。


## 调试建议

- 先单独测试底盘循迹，确认 `BASE_SPEED` 与 `TURN_GAIN` 适配当前底盘。
- 再单独测试舵机角度，确认 `moveLiftToPickupPose()` 和 `closeGripper()` 不会卡死机构。
- 若站点重复计数，优先增大 `MARKER_COOLDOWN_MS` 或延长站点间距。
- 若弯道循迹偏慢或过冲，可重新调整误差放大系数和基础速度。


## 验证标准

| 测试项 | 通过标准 |
| --- | --- |
| 路径跟随 | 小车能沿黑线稳定前进 |
| 站点识别 | 经过宽标记时站点编号只增加一次 |
| 抓取动作 | 到达取货站后能完成夹取并抬升 |
| 放置动作 | 到达卸货站后能完成放置并停止 |


## 可扩展方向

- 加入二维码或 RFID 站点识别，替代单纯的黑色宽标记。
- 增加超声波避障，实现搬运与避障融合。
- 增加多任务路线和多次取放货逻辑。
- 使用上位机记录站点状态与任务完成时间。
